# Wake Word "Letícia" para HA Voice PE (v24 — microWakeWord)

## Reinício consolidado — espelha o `basic_training_notebook.ipynb` oficial de kahrendt/microWakeWord

### Hardware de destino: Home Assistant Voice PE

| Componente | Especificação |
|------------|---------------|
| SoC | ESP32-S3 — 16 MB Flash + 8 MB PSRAM octal |
| DSP de áudio | XMOS XU316 (echo cancellation, noise removal, AGC) |
| Microfones | Dual-mic array interno |
| Engine wake word | **`micro_wake_word`** (ESPHome, on-device) |
| Modelo de áudio | 16 kHz mono, 40 features mel a cada 10 ms |
| Firmware | ESPHome 2024.7+ |

### Correções consolidadas em v24 (lições aprendidas v18→v23)

| # | Origem | Correção |
|---|--------|----------|
| 22a | v22→v23 | `piper` CLI (do pacote `piper-tts`) em vez de `rhasspy/piper-sample-generator` — aceita `.onnx` pt_BR |
| 22b | v22→v23 | `piper-tts` no Etapa 1a (fornece o binário `piper`) |
| 22c | v22→v23 | `sys.executable` em todos os subprocessos Python |
| 22d | v22→v23 | `ThreadPoolExecutor` com workers paralelos para geração |
| 20 | v18→v21 | Auto-resample 22050Hz → 16kHz pós-geração Piper |
| **24a** | **v23→v24** | **CORREÇÃO CRÍTICA**: feature generation via API Python oficial (`Clips`+`Augmentation`+`SpectrogramGeneration`+`RaggedMmap`). v22/v23 tentavam `microwakeword.generate` que **NÃO EXISTE** — falharia na Etapa 6 mesmo se Etapa 3 passasse |
| **24b** | **v23→v24** | Download de MIT RIRs + AudioSet para augmentation de positivos (faltava em v22/v23) |
| **24c** | **v23→v24** | JSON manifest com `"version": 2` (igual aos modelos oficiais okay_nabu/hey_jarvis) |
| **24d** | **v23→v24** | Forks oficiais `puddly/pymicro-features` e `whatsnowplaying/audio-metadata` (conforme notebook oficial) |
| **24e** | **v23→v24** | Splits training/validation/testing com `slide_frames` correto (conforme notebook oficial) |
| **24f** | **v23→v24** | Sample generation gera arquivos numerados (`0.wav`, `1.wav`, …) como espera o `Clips.audio_generator` |

## Instruções de uso
1. **GPU T4 ativa** — Ambiente de execução → Alterar tipo → T4 GPU
2. Execute **Etapa 1a** → clique **Reiniciar sessão** quando aparecer
3. Execute **Etapa 1b em diante** ("Executar tudo a partir daqui")
4. Os arquivos `leticia_mww.tflite` e `leticia_mww.json` serão baixados automaticamente no final

> **Tempo total estimado:** 2-3 horas (Colab T4)

---

## Etapa 0: Diagnóstico GPU

Confirma CUDA disponível antes de instalar.

In [ ]:
import subprocess

print('=== DIAGNÓSTICO GPU ===')
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    for line in r.stdout.split('\n')[:10]:
        print(line)
    print()
    print('✅ GPU disponível')
else:
    print('❌ nvidia-smi não encontrado — ative T4 GPU em Ambiente de execução → Alterar tipo de ambiente')
    raise RuntimeError('GPU obrigatória — selecione T4 GPU')
print('=== FIM DO DIAGNÓSTICO ===')

## Etapa 1a: Instalação das dependências

**⚠️ REINICIE A SESSÃO após esta célula** (Ambiente de execução → Reiniciar sessão).
Depois execute a partir da Etapa 1b.

### Pacotes
- **`pymicro-features`** (fork puddly/minimum-cpp-version) — feature extractor C++
- **`audio-metadata`** (fork whatsnowplaying, sem `attrs` que quebra Jupyter)
- **`microwakeword`** (kahrendt/microWakeWord, instalado editável)
  - Transitivos: `audiomentations`, `datasets`, `mmap_ninja`, `numpy`, `pyyaml`, `tensorflow>=2.16`, `webrtcvad-wheels`, `ai-edge-litert`
- **`torch + torchaudio`** (CUDA 12.1 para T4)
- **`piper-tts`** + **`piper-phonemize-cross==1.2.1`** — CLI `piper` para sintetizar amostras pt_BR

In [ ]:
import subprocess, sys, os, re

print('=' * 60)
print('  ETAPA 1a: Instalação das dependências (v24)')
print('=' * 60)

# Filtro de ruído de avisos do Colab (não são erros)
_RUIDO = re.compile(
    r'timm|fastai|torchvision|protobuf|numpy|jax|rasterio|shap|'
    r'cupy|opencv|tifffile|grain|tobler|ydf|opentelemetry|grpc|'
    r'google-|xarray|pytensor|transformers|gcsfs|fsspec'
)

def pip_install(pkgs, desc='', critico=True):
    """Instala pacotes via pip, distinguindo avisos de erros reais."""
    if isinstance(pkgs, str):
        pkgs = pkgs.split()
    label = desc or pkgs[0]
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *pkgs],
        capture_output=True, text=True
    )
    saida = r.stdout + r.stderr
    erros, avisos, bloco = [], [], False
    for linha in saida.split('\n'):
        l = linha.strip()
        if not l: continue
        if "pip's dependency resolver" in l:
            bloco = True; continue
        if bloco:
            if _RUIDO.search(l): continue
            if 'requires' in l and ('but you have' in l or 'which is not installed' in l):
                avisos.append(l)
            else:
                bloco = False
        if not bloco and l.startswith('ERROR:'):
            erros.append(l)
    icone = '❌' if erros else ('⚠️' if avisos else '✅')
    print(f'  {icone} {label}')
    for a in avisos[:2]: print(f'      ⚠️  {a[:120]}')
    for e in erros: print(f'      ❌ {e[:200]}')
    if erros and critico:
        raise RuntimeError(f'Falha crítica: {erros[0]}')
    return not erros

# ── Fase 1: Clonar microWakeWord ───────────────────────────────
if not os.path.exists('microWakeWord'):
    print('\n[Fase 1] Clonando microWakeWord...')
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/kahrendt/microWakeWord'], check=True)
    print('  ✅ microWakeWord clonado')
else:
    print('  ✅ microWakeWord já existe')

# ── Fase 2: Forks oficiais (FIX 24d) ───────────────────────────
print('\n[Fase 2] Forks oficiais (compatibilidade build C++ e attrs/Jupyter)...')
pip_install(
    ['git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'],
    desc='pymicro-features (puddly fork)'
)
pip_install(
    ['git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'],
    desc='audio-metadata (whatsnowplaying fork)'
)

# ── Fase 3: microwakeword (puxa audiomentations, datasets, mmap_ninja, TF≥2.16, etc.) ──
print('\n[Fase 3] microwakeword + dependências transitivas...')
pip_install(['-e', './microWakeWord'], desc='microwakeword (editable, com deps)')

# ── Fase 4: PyTorch CUDA 12.1 para Colab T4 ────────────────────
print('\n[Fase 4] PyTorch + torchaudio (CUDA 12.1)...')
pip_install(
    ['torch==2.4.0', 'torchaudio==2.4.0',
     '--index-url', 'https://download.pytorch.org/whl/cu121'],
    desc='torch + torchaudio (cu121)'
)

# ── Fase 5: Piper TTS para síntese pt_BR (FIX 22b) ─────────────
print('\n[Fase 5] Piper TTS (CLI `piper` para pt_BR)...')
pip_install(['piper-tts', 'piper-phonemize-cross==1.2.1'], desc='piper-tts + phonemize-cross')

# ── Fase 6: Utilitários (já vêm como transitivos, mas garantimos) ──
print('\n[Fase 6] Utilitários (scipy, tqdm)...')
pip_install(['scipy', 'tqdm'], desc='scipy + tqdm')

print()
print('=' * 60)
print('  ⚠️  REINICIE A SESSÃO AGORA')
print('  Ambiente de execução → Reiniciar sessão (Ctrl+M .)')
print('  Depois execute a partir da Etapa 1b')
print('=' * 60)

## Etapa 1b: Verificação pós-reinício

Confirma que tudo importa corretamente e a GPU está visível.

In [ ]:
import importlib, shutil, os, sys

print('=' * 60)
print('  ETAPA 1b: Verificação do ambiente (v24)')
print('=' * 60)

all_ok = True

# ── Pacotes Python core ─────────────────────────────────────────
core_pkgs = [
    ('torch', '__version__'),
    ('torchaudio', '__version__'),
    ('tensorflow', '__version__'),
    ('microwakeword', None),
    ('audiomentations', '__version__'),
    ('mmap_ninja', None),
    ('datasets', '__version__'),
    ('scipy', '__version__'),
    ('pymicro_features', None),
    ('audio_metadata', None),
    ('webrtcvad', None),
]
print('\n[Pacotes Python]')
for pkg, attr in core_pkgs:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, attr, 'ok') if attr else 'ok'
        print(f'  ✅ {pkg}: {ver}')
    except ImportError as e:
        print(f'  ❌ {pkg}: {e}')
        all_ok = False

# ── API pública do microwakeword (FIX 24a — confere que as classes existem) ──
print('\n[API microwakeword.audio]')
try:
    from microwakeword.audio.clips import Clips
    from microwakeword.audio.augmentation import Augmentation
    from microwakeword.audio.spectrograms import SpectrogramGeneration
    print('  ✅ Clips, Augmentation, SpectrogramGeneration')
except ImportError as e:
    print(f'  ❌ API audio: {e}')
    all_ok = False

try:
    from mmap_ninja.ragged import RaggedMmap
    print('  ✅ RaggedMmap')
except ImportError as e:
    print(f'  ❌ RaggedMmap: {e}')
    all_ok = False

# ── CLI piper (FIX 22b) ────────────────────────────────────────
print('\n[CLI piper]')
piper_path = shutil.which('piper')
if piper_path:
    print(f'  ✅ piper: {piper_path}')
else:
    print('  ❌ piper CLI não encontrado — Etapa 1a falhou ou sessão não reiniciada')
    all_ok = False

# ── GPU ────────────────────────────────────────────────────────
print('\n[GPU CUDA]')
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  ✅ {name} ({mem:.1f} GB)')
    print(f'  ✅ PyTorch: {torch.__version__}')
else:
    print('  ❌ CUDA indisponível — ative T4 GPU')
    all_ok = False

import tensorflow as tf
tf_gpus = tf.config.list_physical_devices('GPU')
if tf_gpus:
    print(f'  ✅ TensorFlow vê {len(tf_gpus)} GPU(s): {tf_gpus[0].name}')
else:
    print('  ⚠️  TensorFlow não vê GPU (treino será CPU — muito lento)')

# ── microWakeWord clonado ───────────────────────────────────────
print('\n[Diretórios]')
print(f'  {"✅" if os.path.exists("microWakeWord") else "❌"} microWakeWord/')

if not all_ok:
    raise RuntimeError('Corrija os erros acima — execute Etapa 1a e reinicie a sessão')
print('\n[OK] ETAPA 1b CONCLUÍDA!')

## Etapa 2: Download de vozes Piper pt_BR

Baixa `pt_BR-faber-medium.onnx` (voz masculina) e `pt_BR-edresson-low.onnx` (voz adicional para variedade).
Ambas geram em 22050Hz — o resample para 16kHz acontece na Etapa 3 (FIX 20).

In [ ]:
import os, subprocess
from IPython.display import Audio, display

print('=' * 60)
print('  ETAPA 2: Vozes Piper pt_BR')
print('=' * 60)

os.makedirs('piper_voices_ptbr', exist_ok=True)

HF_BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
VOICES = [
    ('pt_BR-faber-medium',  'pt/pt_BR/faber/medium'),
    ('pt_BR-edresson-low',  'pt/pt_BR/edresson/low'),
]

for name, path in VOICES:
    for ext in ['.onnx', '.onnx.json']:
        dest = f'piper_voices_ptbr/{name}{ext}'
        if not os.path.exists(dest):
            url = f'{HF_BASE}/{path}/{name}{ext}'
            print(f'  Baixando {name}{ext}...')
            subprocess.run(['wget', '-q', '--show-progress', '-O', dest, url], check=True)
        size_mb = os.path.getsize(dest) / 1024**2
        print(f'  ✅ {name}{ext} ({size_mb:.1f} MB)')

# ── Teste de pronúncia (1 clip por voz) ────────────────────────
print('\n[2b] Testando pronúncia de "letícia":')
os.makedirs('test_audio', exist_ok=True)
for name, _ in VOICES:
    voice = f'piper_voices_ptbr/{name}.onnx'
    out = f'test_audio/test_{name}.wav'
    r = subprocess.run(
        ['piper', '--model', voice, '--output_file', out],
        input='letícia', capture_output=True, text=True
    )
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print(f'  ✅ {name}:')
        display(Audio(out, autoplay=False))
    else:
        print(f'  ❌ {name} falhou: {r.stderr[:200]}')

print('\n[OK] ETAPA 2 CONCLUÍDA!')

## Etapa 3: Gerar amostras TTS com `piper` CLI

**FIX 22a** + **22d** + **24f**: Gera 1500 amostras de "letícia" via `piper` CLI (paralelo, 4 workers),
salvando como `0.wav`, `1.wav`, … (formato esperado pelo `Clips` da microwakeword).

**FIX 20**: Resample automático 22050Hz → 16kHz pós-geração.

In [ ]:
import os, subprocess, random, time, torchaudio
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

print('=' * 60)
print('  ETAPA 3: Geração de amostras TTS (piper CLI)')
print('=' * 60)

# ── Configuração ────────────────────────────────────────────────
TARGET_WORD = 'letícia'
N_SAMPLES   = 1500     # Mais amostras = modelo melhor
SAMPLES_DIR = 'generated_samples'
N_WORKERS   = 4

VOICES = [
    'piper_voices_ptbr/pt_BR-faber-medium.onnx',
    'piper_voices_ptbr/pt_BR-edresson-low.onnx',
]
VOICES = [v for v in VOICES if os.path.exists(v)]
if not VOICES:
    raise FileNotFoundError('Nenhuma voz pt_BR disponível — execute Etapa 2')

# Variabilidade nos parâmetros do Piper (cobre maior gama acústica)
LENGTH_SCALES = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25]
NOISE_SCALES  = [0.50, 0.60, 0.667, 0.70, 0.80, 0.90, 0.98]
NOISE_WS      = [0.50, 0.60, 0.70, 0.80, 0.90, 0.98]

os.makedirs(SAMPLES_DIR, exist_ok=True)

# FIX 24f: arquivos numerados 0.wav, 1.wav, ... (formato esperado por Clips)
def existing_count():
    return len([f for f in os.listdir(SAMPLES_DIR) if f.endswith('.wav')])

existing = existing_count()

# ── Função de geração de 1 clip ────────────────────────────────
def gen_one(idx):
    voice = random.choice(VOICES)
    out = f'{SAMPLES_DIR}/{idx}.wav'
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return True
    try:
        # FIX 22a: piper CLI aceita .onnx pt_BR nativamente, sem flag --cuda
        r = subprocess.run(
            ['piper',
             '--model',        voice,
             '--output_file',  out,
             '--length-scale', str(random.choice(LENGTH_SCALES)),
             '--noise-scale',  str(random.choice(NOISE_SCALES)),
             '--noise-w',      str(random.choice(NOISE_WS))],
            input=TARGET_WORD,
            capture_output=True, text=True, timeout=30
        )
        return os.path.exists(out) and os.path.getsize(out) > 0
    except Exception:
        return False

# ── Loop paralelo ──────────────────────────────────────────────
if existing >= N_SAMPLES:
    print(f'  ✅ {existing} amostras já existem — pulando geração')
else:
    needed_indices = list(range(existing, N_SAMPLES))
    print(f'\n[3a] Gerando {len(needed_indices)} amostras com {N_WORKERS} workers...')

    t0 = time.time()
    success = 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(gen_one, i): i for i in needed_indices}
        pbar = tqdm(total=len(needed_indices), desc='Piper TTS', unit='clip')
        for fut in as_completed(futures):
            if fut.result(): success += 1
            pbar.update(1)
        pbar.close()

    elapsed = time.time() - t0
    total_now = existing_count()
    print(f'  ✅ {success}/{len(needed_indices)} gerados em {elapsed/60:.1f} min')
    print(f'  Total no diretório: {total_now}/{N_SAMPLES}')

# ── FIX 20: Resample 22050Hz → 16kHz ────────────────────────────
print('\n[FIX 20] Verificando sample rate (faber-medium = 22050Hz)...')
wavs = sorted(
    [f for f in os.listdir(SAMPLES_DIR) if f.endswith('.wav')],
    key=lambda x: int(x.split('.')[0]) if x.split('.')[0].isdigit() else 0
)
bad = []
for f in wavs:
    try:
        info = torchaudio.info(f'{SAMPLES_DIR}/{f}')
        if info.sample_rate != 16000:
            bad.append((f, info.sample_rate))
    except Exception:
        pass

if bad:
    print(f'  ⚠️  {len(bad)} clips em {bad[0][1]}Hz → convertendo para 16kHz...')
    for fname, sr in tqdm(bad, desc='Resample 16kHz'):
        path = f'{SAMPLES_DIR}/{fname}'
        wf, _ = torchaudio.load(path)
        if wf.shape[0] > 1:
            wf = wf.mean(dim=0, keepdim=True)
        wf = torchaudio.transforms.Resample(sr, 16000)(wf)
        torchaudio.save(path, wf, 16000)
    print(f'  ✅ {len(bad)} clips convertidos')
else:
    print(f'  ✅ Todos os {len(wavs)} clips já estão em 16kHz')

# ── Validação final ────────────────────────────────────────────
valid = sum(1 for f in wavs
            if torchaudio.info(f'{SAMPLES_DIR}/{f}').sample_rate == 16000)
print(f'\n  ✅ {valid}/{len(wavs)} amostras válidas (16kHz mono)')
if valid < N_SAMPLES * 0.9:
    raise RuntimeError(f'Apenas {valid}/{N_SAMPLES} amostras válidas — verifique piper')

print('\n[OK] ETAPA 3 CONCLUÍDA!')

## Etapa 4: Download de dados de augmentation + datasets negativos

**FIX 24b**: Baixa **MIT RIRs** + **AudioSet** (para augmentation das amostras positivas) +
**negative_datasets** pré-processados de kahrendt (para o treinamento).

| Dado | Função | Tamanho |
|------|--------|---------|
| `mit_rirs/` | Impulse responses (reverberação) | ~30 MB |
| `audioset_16k/` | Ruído de fundo (música, ambiente) | ~1 GB |
| `negative_datasets/speech/` | Fala genérica (spectrograms) | ~3 GB |
| `negative_datasets/dinner_party/` | Conversas (spectrograms) | ~3 GB |
| `negative_datasets/dinner_party_eval/` | Validation set | ~500 MB |
| `negative_datasets/no_speech/` | Sem fala (spectrograms) | ~3 GB |

In [ ]:
import os, subprocess, zipfile, datasets, scipy.io.wavfile as wav
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

print('=' * 60)
print('  ETAPA 4: Augmentation data + Negative datasets')
print('=' * 60)

# ── 4a: MIT Room Impulse Responses ──────────────────────────────
rir_dir = 'mit_rirs'
if not os.path.exists(rir_dir) or len([f for f in os.listdir(rir_dir) if f.endswith('.wav')]) == 0:
    print('\n[4a] MIT Room Impulse Responses (~30 MB)...')
    os.makedirs(rir_dir, exist_ok=True)
    rir_ds = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    count = 0
    for row in tqdm(rir_ds, desc='MIT RIRs'):
        name = row['audio']['path'].split('/')[-1]
        arr = np.array(row['audio']['array'])
        wav.write(os.path.join(rir_dir, name), 16000, (arr * 32767).astype(np.int16))
        count += 1
    print(f'  ✅ MIT RIRs: {count} arquivos')
else:
    n = len([f for f in os.listdir(rir_dir) if f.endswith('.wav')])
    print(f'  ✅ MIT RIRs: {n} arquivos (já existem)')

# ── 4b: AudioSet background ─────────────────────────────────────
as_dir = 'audioset_16k'
if not os.path.exists(as_dir) or len(os.listdir(as_dir)) == 0:
    print('\n[4b] AudioSet background (~1 GB)...')
    os.makedirs('audioset', exist_ok=True)
    os.makedirs(as_dir, exist_ok=True)
    fname = 'bal_train09.tar'
    if not os.path.exists(f'audioset/{fname}'):
        link = f'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}'
        subprocess.run(['wget', '-q', '--show-progress', '-O',
                        f'audioset/{fname}', link], check=True)
    subprocess.run(['tar', '-xf', f'audioset/{fname}', '-C', 'audioset'],
                   capture_output=True)
    flacs = list(Path('audioset/audio').glob('**/*.flac')) if os.path.exists('audioset/audio') else []
    if flacs:
        print(f'  Convertendo {len(flacs)} FLAC → WAV 16kHz...')
        as_ds = datasets.Dataset.from_dict({'audio': [str(p) for p in flacs]})
        as_ds = as_ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
        for row in tqdm(as_ds, desc='AudioSet'):
            name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
            arr = np.array(row['audio']['array'])
            wav.write(os.path.join(as_dir, name), 16000, (arr * 32767).astype(np.int16))
    n = len(os.listdir(as_dir))
    print(f'  ✅ AudioSet: {n} clips')
else:
    print(f'  ✅ AudioSet: {len(os.listdir(as_dir))} clips (já existem)')

# ── 4c: Datasets negativos pré-processados (kahrendt) ──────────
print('\n[4c] Negative datasets (spectrograms pré-gerados)...')
NEG_BASE = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main'
NEG_NAMES = ['dinner_party', 'dinner_party_eval', 'speech', 'no_speech']

os.makedirs('negative_datasets', exist_ok=True)
for name in NEG_NAMES:
    dest_dir = f'negative_datasets/{name}'
    if os.path.exists(dest_dir) and len(os.listdir(dest_dir)) > 0:
        print(f'  ✅ {name}: já existe ({len(os.listdir(dest_dir))} itens)')
        continue

    zip_path = f'{name}.zip'
    if not os.path.exists(zip_path):
        print(f'\n  [{name}] Baixando...')
        subprocess.run(['wget', '-q', '--show-progress', '-c',
                        f'{NEG_BASE}/{name}.zip', '-O', zip_path], check=True)
    print(f'  [{name}] Extraindo...')
    os.makedirs(dest_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest_dir)
    print(f'  ✅ {name}: {len(os.listdir(dest_dir))} itens')

# ── Resumo ──────────────────────────────────────────────────────
print('\nResumo dos dados:')
for d, label in [('mit_rirs', 'MIT RIRs'),
                 ('audioset_16k', 'AudioSet'),
                 ('generated_samples', 'Positivos (letícia)')]:
    n = len([f for f in os.listdir(d) if f.endswith('.wav')]) if os.path.exists(d) else 0
    status = '✅' if n > 0 else '❌'
    print(f'  {status} {label}: {n} wavs')
for name in NEG_NAMES:
    d = f'negative_datasets/{name}'
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    status = '✅' if n > 0 else '❌'
    print(f'  {status} negative/{name}: {n} itens')

print('\n[OK] ETAPA 4 CONCLUÍDA!')

## Etapa 5: Augmentar amostras + Gerar spectrograms

**FIX 24a (CRÍTICO)**: Usa a **API Python oficial** do microwakeword (NÃO CLI, que não existe).

Cria `generated_augmented_features/{training,validation,testing}/wakeword_mmap/` —
estrutura RaggedMmap que o `model_train_eval` espera.

- **training**: 2x repetições, slide_frames=10 (simula streaming)
- **validation**: 1x, slide_frames=10
- **testing**: 1x, slide_frames=1 (sem repetição, modelo streaming)

In [ ]:
import os
from IPython.display import Audio, display

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from microwakeword.audio.audio_utils import save_clip
from mmap_ninja.ragged import RaggedMmap

print('=' * 60)
print('  ETAPA 5: Augmentation + Spectrogram features (API oficial)')
print('=' * 60)

# ── 5a: Configurar Clips ───────────────────────────────────────
print('\n[5a] Configurando Clips (input: generated_samples)...')
clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,            # 10% para validação + testing
)
print('  ✅ Clips configurado')

# ── 5b: Configurar Augmentation ────────────────────────────────
# Backgrounds: filtra apenas pastas que existem
bg_dirs = [d for d in ['audioset_16k', 'fma_16k'] if os.path.exists(d) and len(os.listdir(d)) > 0]
rir_dirs = [d for d in ['mit_rirs'] if os.path.exists(d) and len(os.listdir(d)) > 0]

print(f'\n[5b] Augmentation — RIRs={rir_dirs} backgrounds={bg_dirs}')
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion':        0.1,
        'PitchShift':            0.1,
        'BandStopFilter':        0.1,
        'AddColorNoise':         0.1,
        'AddBackgroundNoise':    0.75 if bg_dirs else 0.0,
        'Gain':                  1.0,
        'RIR':                   0.5 if rir_dirs else 0.0,
    },
    impulse_paths=rir_dirs,
    background_paths=bg_dirs,
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)
print('  ✅ Augmentation configurado')

# ── 5c: Preview de 1 clip augmentado ───────────────────────────
print('\n[5c] Preview de clip augmentado:')
try:
    random_clip = clips.get_random_clip()
    augmented = augmenter.augment_clip(random_clip)
    save_clip(augmented, 'augmented_preview.wav')
    display(Audio('augmented_preview.wav', autoplay=False))
    print('  ✅ Preview gerado')
except Exception as e:
    print(f'  ⚠️  Preview falhou: {e} (não-fatal)')

# ── 5d: Gerar spectrograms para training/validation/testing ─────
OUTPUT_DIR = 'generated_augmented_features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('\n[5d] Gerando spectrograms ragged_mmap...')

# Mapa split → (split_name interno do Clips, repetition, slide_frames)
SPLITS = [
    ('training',   'train',      2, 10),
    ('validation', 'validation', 1, 10),
    ('testing',    'test',       1, 1),   # streaming → slide_frames=1
]

for split, split_name, repetition, slide_frames in SPLITS:
    split_dir = os.path.join(OUTPUT_DIR, split)
    mmap_dir = os.path.join(split_dir, 'wakeword_mmap')

    if os.path.exists(mmap_dir) and len(os.listdir(mmap_dir)) > 0:
        print(f'  ✅ {split}: já existe — pulando')
        continue

    os.makedirs(split_dir, exist_ok=True)
    print(f'\n  [{split}] split_name={split_name}, repetition={repetition}, slide_frames={slide_frames}')

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide_frames,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=mmap_dir,
        sample_generator=spectrograms.spectrogram_generator(
            split=split_name, repeat=repetition
        ),
        batch_size=100,
        verbose=True,
    )
    print(f'  ✅ {split} gerado em {mmap_dir}')

# ── Verificação ────────────────────────────────────────────────
print('\nVerificação dos features gerados:')
for split, *_ in SPLITS:
    mmap_dir = os.path.join(OUTPUT_DIR, split, 'wakeword_mmap')
    if os.path.exists(mmap_dir):
        items = os.listdir(mmap_dir)
        print(f'  ✅ {split}/wakeword_mmap: {len(items)} arquivos')
    else:
        print(f'  ❌ {split}: FALTANDO')

print('\n[OK] ETAPA 5 CONCLUÍDA!')

## Etapa 6: Configuração do treinamento (YAML)

Cria `training_parameters.yaml` com os hiperparâmetros do modelo MixedNet.
Os valores espelham o `basic_training_notebook.ipynb` oficial.

In [ ]:
import yaml, os

print('=' * 60)
print('  ETAPA 6: training_parameters.yaml')
print('=' * 60)

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',

    # Cada features_dir deve ter subpastas training/ validation/ testing/
    # contendo ragged_mmap_folders_ending_in_mmap
    'features': [
        {
            'features_dir': 'generated_augmented_features',   # gerado na Etapa 5
            'sampling_weight': 2.0,
            'penalty_weight':  1.0,
            'truth':           True,
            'truncation_strategy': 'truncate_start',
            'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/speech',
            'sampling_weight': 10.0,
            'penalty_weight':  1.0,
            'truth':           False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/dinner_party',
            'sampling_weight': 10.0,
            'penalty_weight':  1.0,
            'truth':           False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/no_speech',
            'sampling_weight': 5.0,
            'penalty_weight':  1.0,
            'truth':           False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },
        {   # Apenas para validation/testing (sampling_weight=0)
            'features_dir': 'negative_datasets/dinner_party_eval',
            'sampling_weight': 0.0,
            'penalty_weight':  1.0,
            'truth':           False,
            'truncation_strategy': 'split',
            'type': 'mmap',
        },
    ],

    'training_steps':        [10000],        # Aumente para 20k-50k para melhor qualidade
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates':        [0.001],
    'batch_size':            128,

    # SpecAugment desligado (config default kahrendt)
    'time_mask_max_size':    [0],
    'time_mask_count':       [0],
    'freq_mask_max_size':    [0],
    'freq_mask_count':       [0],

    'eval_step_interval':  500,
    'clip_duration_ms':    1500,
    'target_minimization': 0.9,
    'minimization_metric': None,
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print('training_parameters.yaml criado:\n')
print(yaml.dump(config, default_flow_style=False, allow_unicode=True, sort_keys=False))
print('[OK] ETAPA 6 CONCLUÍDA!')

## Etapa 7: Treinamento

**~30-60 min** com T4 GPU para 10k steps. Pode demorar mais para 50k.

Comando idêntico ao `basic_training_notebook.ipynb` oficial — arquitetura `mixednet` com 4 blocos.

> No Colab, as mini-batch progressions não aparecem; pode parecer travado por vários minutos.

In [ ]:
import subprocess, sys, os, glob

print('=' * 60)
print('  ETAPA 7: Treinamento microWakeWord (mixednet)')
print('=' * 60)

# FIX 22c: sys.executable em vez de "python"
cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config=training_parameters.yaml',
    '--train', '1',
    '--restore_checkpoint', '1',
    '--test_tf_nonstreaming',              '0',
    '--test_tflite_nonstreaming',          '0',
    '--test_tflite_nonstreaming_quantized', '0',
    '--test_tflite_streaming',             '0',
    '--test_tflite_streaming_quantized',   '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters',    '64,64,64,64',
    '--repeat_in_block',      '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection',  '0,0,0,0',
    '--first_conv_filters',   '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]

print('Executando:\n  ' + ' \\\n    '.join(cmd) + '\n')
subprocess.run(cmd, check=True)

# ── Verificar output ────────────────────────────────────────────
TFLITE_DEFAULT = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
tflite_path = TFLITE_DEFAULT

if not os.path.exists(tflite_path):
    print('\n⚠️  TFLite não no path padrão — procurando...')
    found = glob.glob('trained_models/**/*.tflite', recursive=True)
    if found:
        tflite_path = sorted(found, key=os.path.getmtime)[-1]
        print(f'  Encontrado: {tflite_path}')

if os.path.exists(tflite_path):
    size_kb = os.path.getsize(tflite_path) / 1024
    print(f'\n  ✅ Modelo: {tflite_path} ({size_kb:.1f} KB)')
else:
    raise FileNotFoundError('Modelo .tflite não gerado — veja logs do training acima')

print('\n[OK] ETAPA 7 CONCLUÍDA!')

## Etapa 8: Manifest JSON + Download

**FIX 24c**: Manifest com `"version": 2` (igual aos modelos oficiais `okay_nabu`, `hey_jarvis`).

### Deploy no HA Voice PE (3 dispositivos)

1. Suba `leticia_mww.tflite` em um GitHub Release
2. Edite `leticia_mww.json` → atualize `"model"` com a URL do Release
3. Suba `leticia_mww.json` também no Release
4. No ESPHome do HA Voice PE, adicione:

   ```yaml
   micro_wake_word:
     models:
       - model: https://github.com/visaodeempresa/ha-wakeword-leticia/releases/download/v1.0-mww/leticia_mww.json
         id: leticia
   ```
5. Compile + OTA → "Letícia" aparece como wake word no HA

In [ ]:
import os, json, shutil, glob

print('=' * 60)
print('  ETAPA 8: Empacotamento + Manifest + Download')
print('=' * 60)

# ── Localizar tflite ────────────────────────────────────────────
tflite_src = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
if not os.path.exists(tflite_src):
    found = glob.glob('trained_models/**/*.tflite', recursive=True)
    if found:
        tflite_src = sorted(found, key=os.path.getmtime)[-1]
    else:
        raise FileNotFoundError('Nenhum .tflite — execute Etapa 7')

os.makedirs('output', exist_ok=True)
tflite_dest = 'output/leticia_mww.tflite'
shutil.copy(tflite_src, tflite_dest)
size_kb = os.path.getsize(tflite_dest) / 1024
print(f'  ✅ {tflite_dest} ({size_kb:.1f} KB)')

# ── Manifest JSON (FIX 24c: version=2) ──────────────────────────
GITHUB_RELEASE_URL = (
    'https://github.com/visaodeempresa/ha-wakeword-leticia'
    '/releases/download/v1.0-mww/leticia_mww.tflite'
)

manifest = {
    'type':              'micro',
    'wake_word':         'Letícia',
    'author':            'Maycon Willian',
    'website':           'https://github.com/visaodeempresa/ha-wakeword-leticia',
    'model':             GITHUB_RELEASE_URL,
    'trained_languages': ['pt'],
    'version':           2,           # ← FIX 24c (era 1 em v22)
    'micro': {
        'probability_cutoff':       0.97,    # mesmo valor de okay_nabu/hey_jarvis
        'sliding_window_size':      5,
        'feature_step_size':        10,
        'tensor_arena_size':        26080,   # 26 KB — confortável no ESP32-S3 com 8MB PSRAM
        'minimum_esphome_version':  '2024.7.0',
    },
}

json_dest = 'output/leticia_mww.json'
with open(json_dest, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f'  ✅ {json_dest}')
print('\n  Manifest gerado:')
print('  ' + json.dumps(manifest, ensure_ascii=False, indent=2).replace('\n', '\n  '))

print('''
╔══════════════════════════════════════════════════════════════╗
║         PRÓXIMOS PASSOS — Deploy nos 3 HA Voice PE          ║
╚══════════════════════════════════════════════════════════════╝

1. CRIAR RELEASE no GitHub:
   https://github.com/visaodeempresa/ha-wakeword-leticia/releases/new
   Tag: v1.0-mww
   Upload os 2 arquivos: leticia_mww.tflite + leticia_mww.json
   Copie a URL pública do JSON

2. EDITAR ESPHome dos HA Voice PE (1 por 1):
   Configurações → ESPHome → (dispositivo) → Editar
   Adicione em micro_wake_word → models:

     - model: https://github.com/visaodeempresa/ha-wakeword-leticia/releases/download/v1.0-mww/leticia_mww.json
       id: leticia

3. INSTALAR via OTA (botão Instalar no ESPHome)
   Cada dispositivo recebe o modelo via wifi automaticamente.

4. No HA: Configurações → Voice Assistants → (pipeline) → Wake word: Letícia
''')

# ── Download automático ─────────────────────────────────────────
try:
    from google.colab import files
    print('Baixando arquivos para sua máquina...')
    for f in [tflite_dest, json_dest]:
        try:
            files.download(f)
        except Exception as e:
            print(f'  ⚠️  Download falhou para {f}: {e}')
            print(f'     Baixe manualmente pelo painel Arquivos (📁) à esquerda')
except ImportError:
    print('Ambiente não-Colab — baixe manualmente:')
    print(f'  {tflite_dest}')
    print(f'  {json_dest}')

print('\n🎉 [OK] ETAPA 8 CONCLUÍDA — TREINAMENTO COMPLETO!')